# Gas Concentration Visualization and Projection

This notebook visualizes weekly gas concentration data (CO2, N2O, CH4) for different experiment conditions and projects future concentrations using best-fit curves.

In [ ]:
# !pip install pandas numpy bokeh scipy

In [11]:
import pandas as pd
import numpy as np
from bokeh.plotting import figure, show, output_notebook
from bokeh.layouts import gridplot
from bokeh.models import HoverTool, Legend
from bokeh.palettes import Category10
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings('ignore')
import glob
import os
import astropy


# Enable Bokeh in notebook
output_notebook()

Loading BokehJS ...

## 1) Loading in data

In [3]:
# 1a) getting in all csvs from  cwd. making it generalizable just in case
csv_files = glob.glob('*.csv')
print(f"Found CSV files: {csv_files}")

# 1b) load + combine
all_data = []
for file in csv_files:
    df = pd.read_csv(file)
    all_data.append(df)

# 1c) combine all dataframes and print to make sure
combined_data = pd.concat(all_data, ignore_index=True)
print(f"Combined data shape: {combined_data.shape}")
print(f"Columns: {combined_data.columns.tolist()}")
print(f"\nFirst few rows:")
combined_data.head()

Found CSV files: ['exp02_data.csv', 'gas_concentration_projections.csv', 'exp000_data.csv', 'exp01_data.csv']
Combined data shape: (1950, 11)
Columns: ['Sample ID', 'Calc Concentration (ppm)', 'days_since_start', 'Gas', 'Water Activity', 'Moles', 'experiment_condition', 'projected_day', 'projected_concentration_ppm', 'model_type', 'r_squared']

First few rows:


,Sample ID,Calc Concentration (ppm),days_since_start,Gas,Water Activity,Moles,experiment_condition,projected_day,projected_concentration_ppm,model_type,r_squared
0,exp_01_01_01,3464.376543,0.0,CO2,0.996141,1.440172e-05,NaN,NaN,NaN,NaN,NaN
1,exp_01_01_01,0.978286,0.0,N2O,0.996141,4.066820e-09,NaN,NaN,NaN,NaN,NaN
2,exp_01_01_01,43.685798,0.0,CH4,0.996141,1.816057e-07,NaN,NaN,NaN,NaN,NaN
3,exp_01_01_02,3658.669350,0.0,CO2,0.996141,1.520941e-05,NaN,NaN,NaN,NaN,NaN
4,exp_01_01_02,0.979812,0.0,N2O,0.996141,4.073164e-09,NaN,NaN,NaN,NaN,NaN


## 2) Getting exp conditions, times, etc.

In [4]:
# 2a) function to get experiment ID
def parse_experiment_id(sample_id):
    """
    Parse experiment ID from sample ID format: exp_XX_YY_ZZ
    Returns experiment condition as exp_XX_YY
    """
    # Handle NaN values (which appear as float)
    if pd.isna(sample_id) or not isinstance(sample_id, str):
        return 'unknown'
    
    parts = sample_id.split('_')
    if len(parts) >= 4:
        return f"{parts[0]}_{parts[1]}_{parts[2]}"
    return sample_id

# 2b) adding experiment condition column for easier sorting (ignoring the replicate number)
combined_data['experiment_condition'] = combined_data['Sample ID'].apply(parse_experiment_id)

# 2c) calling 2a, and printing outputs
unique_conditions = sorted(combined_data['experiment_condition'].unique())
print(f"Unique experiment conditions: {unique_conditions}")
print(f"Number of conditions: {len(unique_conditions)}")

# 2d) getting time points and gases (just in case we want to add more gases)
time_points = sorted(combined_data['days_since_start'].unique())
gases = sorted(combined_data['Gas'].unique())
print(f"Time points (days): {time_points}")
print(f"Gases: {gases}")

Unique experiment conditions: ['exp_01_01', 'exp_01_02', 'exp_01_03', 'exp_01_04', 'exp_01_05', 'exp_01_06', 'unknown']
Number of conditions: 7
Time points (days): [np.float64(0.0), np.float64(7.0), np.float64(13.0), np.float64(21.0), np.float64(28.0), np.float64(35.0), np.float64(42.0), np.float64(nan)]
Gases: ['CH4', 'CO2', 'N2O']


## 3) Chat-assisted debugging

In [5]:
# Debug: Check data before groupby operation
print("=== DEBUGGING BEFORE GROUPBY ===")
print(f"Combined data shape: {combined_data.shape}")
print(f"Combined data columns: {combined_data.columns.tolist()}")
print(f"Data types:\n{combined_data.dtypes}")

# Check if required columns exist
required_columns = ['experiment_condition', 'days_since_start', 'Gas', 'Calc Concentration (ppm)', 'Water Activity']
missing_columns = [col for col in required_columns if col not in combined_data.columns]
if missing_columns:
    print(f"Missing columns: {missing_columns}")
    
    # If experiment_condition is missing, create it here
    if 'experiment_condition' in missing_columns and 'Sample ID' in combined_data.columns:
        print("Creating experiment_condition column...")
        def parse_experiment_id(sample_id):
            parts = sample_id.split('_')
            if len(parts) >= 4:
                return f"{parts[0]}_{parts[1]}_{parts[2]}"
            return sample_id
        
        combined_data['experiment_condition'] = combined_data['Sample ID'].apply(parse_experiment_id)
        print("experiment_condition column created successfully")

print(f"Sample of data:\n{combined_data.head()}")

# Calculate mean concentration for each condition, time point, and gas
# (averaging across replicates within each condition)
if len(combined_data) > 0:
    mean_data = combined_data.groupby(['experiment_condition', 'days_since_start', 'Gas']).agg({
        'Calc Concentration (ppm)': ['mean', 'std'],
        'Water Activity': 'mean'
    }).reset_index()

    # Flatten column names
    mean_data.columns = ['experiment_condition', 'days_since_start', 'Gas', 'mean_conc', 'std_conc', 'mean_water_activity']

    print(f"Mean data shape: {mean_data.shape}")
    print("Mean data sample:")
    display(mean_data.head())
else:
    print("No data available for groupby operation")
    mean_data = pd.DataFrame()

=== DEBUGGING BEFORE GROUPBY ===
Combined data shape: (1950, 11)
Combined data columns: ['Sample ID', 'Calc Concentration (ppm)', 'days_since_start', 'Gas', 'Water Activity', 'Moles', 'experiment_condition', 'projected_day', 'projected_concentration_ppm', 'model_type', 'r_squared']
Data types:
Sample ID                       object
Calc Concentration (ppm)       float64
days_since_start               float64
Gas                             object
Water Activity                 float64
Moles                          float64
experiment_condition            object
projected_day                  float64
projected_concentration_ppm    float64
model_type                      object
r_squared                      float64
dtype: object
Sample of data:
      Sample ID  Calc Concentration (ppm)  days_since_start  Gas  \
0  exp_01_01_01               3464.376543               0.0  CO2   
1  exp_01_01_01                  0.978286               0.0  N2O   
2  exp_01_01_01                 43.685798 

,experiment_condition,days_since_start,Gas,mean_conc,std_conc,mean_water_activity
0,exp_01_01,0.0,CH4,51.330700,23.303898,0.996141
1,exp_01_01,0.0,CO2,3786.347480,1224.949320,0.996141
2,exp_01_01,0.0,N2O,0.978991,0.000681,0.996141
3,exp_01_01,7.0,CH4,1129.136170,273.598699,0.996141
4,exp_01_01,7.0,CO2,8832.409226,3126.227263,0.996141


## 4) setting up some curve fitting functions (we can add more, keeping it simple)
It should select the one with the best r^2 :)

In [6]:
# 4a) defining curve fitting functions
def exponential_growth(x, a, b, c):
    """Exponential growth: a * exp(b * x) + c"""
    return a * np.exp(b * x) + c

def linear_model(x, a, b):
    """Linear model: a * x + b"""
    return a * x + b

def logistic_growth(x, a, b, c, d):
    """Logistic growth: a / (1 + exp(-b * (x - c))) + d"""
    return a / (1 + np.exp(-b * (x - c))) + d

# 4b) function to fit the best curve, and keep it
def fit_best_curve(x_data, y_data):
    """
    Try different curve fitting models and return the best fit
    """
    models = {
        'linear': (linear_model, [1, 1]),
        'exponential': (exponential_growth, [1, 0.1, 1]),
        'logistic': (logistic_growth, [1000, 0.1, 20, 0])
    }
    
    best_model = None
    best_params = None
    best_r2 = -np.inf
    best_name = 'linear'
    
    for name, (func, p0) in models.items():
        try:
            # Fit the curve
            popt, _ = curve_fit(func, x_data, y_data, p0=p0, maxfev=1000)
            
            # Calculate R-squared
            y_pred = func(x_data, *popt)
            ss_res = np.sum((y_data - y_pred) ** 2)
            ss_tot = np.sum((y_data - np.mean(y_data)) ** 2)
            r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
            
            if r2 > best_r2:
                best_model = func
                best_params = popt
                best_r2 = r2
                best_name = name
                
        except Exception as e:
            continue
    
    return best_model, best_params, best_r2, best_name

## 5) Setting up which experiments we want to plot :)
defaults to first 6, but you can of course change it.
Also, you can change projection time to be longer, e.g. change number of days from 60--> 100

In [7]:
# 5a) Select 6 experiment conditions for 3x2 grid
selected_conditions = unique_conditions[:6]
print(f"Selected conditions for visualization: {selected_conditions}")

#5b) set these projection time points
total_days = 60  # You can change this to extend the projection
number_of_points = 120  # Number of points for smooth projection, should be ~2x days


# 5c) Create projection time points (extend to 60 days)
projection_days = np.linspace(0, total_days, number_of_points)

# 5d) colour palette, plotting each condition, etc. etc.
gas_colors = {'CO2': '#1f77b4', 'N2O': '#ff7f0e', 'CH4': '#2ca02c'}
plots = []

for condition in selected_conditions:
    # Create figure for this condition
    p = figure(
        title=f"Gas Concentrations - {condition}",
        x_axis_label="Days since start",
        y_axis_label="Concentration (ppm)",
        width=400,
        height=300,
        tools="pan,wheel_zoom,box_zoom,reset,save"
    )
    
    # Add hover tool
    hover = HoverTool(tooltips=[
        ("Gas", "@gas"),
        ("Day", "@x"),
        ("Concentration", "@y{0.00} ppm"),
        ("Model", "@model")
    ])
    p.add_tools(hover)
    
    # Filter data for this condition
    condition_data = mean_data[mean_data['experiment_condition'] == condition]
    
    legend_items = []
    
    # Plot each gas
    for gas in gases:
        gas_data = condition_data[condition_data['Gas'] == gas]
        
        if len(gas_data) > 0:
            x_data = gas_data['days_since_start'].values
            y_data = gas_data['mean_conc'].values
            y_err = gas_data['std_conc'].values
            
            # Plot observed data points
            observed = p.circle(
                x_data, y_data,
                size=8,
                color=gas_colors[gas],
                alpha=0.8,
                legend_label=f"{gas} (observed)"
            )
            
            # Add error bars if available
            if not np.isnan(y_err).all():
                p.segment(x_data, y_data - y_err, x_data, y_data + y_err, 
                         line_color=gas_colors[gas], line_alpha=0.5)
            
            # Fit curve and project
            if len(x_data) >= 3:  # Need at least 3 points for curve fitting
                try:
                    best_model, best_params, r2, model_name = fit_best_curve(x_data, y_data)
                    
                    if best_model is not None:
                        # Generate projection
                        y_projection = best_model(projection_days, *best_params)
                        
                        # Ensure no negative concentrations
                        y_projection = np.maximum(y_projection, 0)
                        
                        # Plot projection
                        projected = p.line(
                            projection_days, y_projection,
                            line_color=gas_colors[gas],
                            line_dash='dashed',
                            line_width=2,
                            alpha=0.7,
                            legend_label=f"{gas} ({model_name}, R²={r2:.3f})"
                        )
                        
                        # Add source data for hover tool
                        p.add_glyph(projected.glyph.data_source, projected.glyph, 
                                   name=f"{gas}_projection")
                        
                except Exception as e:
                    print(f"Could not fit curve for {condition} - {gas}: {e}")
    
    # Customize legend
    p.legend.location = "top_left"
    p.legend.click_policy = "hide"
    p.legend.label_text_font_size = "8pt"
    
    plots.append(p)

print(f"Created {len(plots)} plots")

Selected conditions for visualization: ['exp_01_01', 'exp_01_02', 'exp_01_03', 'exp_01_04', 'exp_01_05', 'exp_01_06']
Could not fit curve for exp_01_01 - CH4: unexpected attribute 'data_source' to Line, possible attributes are decorations, js_event_callbacks, js_property_callbacks, line_alpha, line_cap, line_color, line_dash, line_dash_offset, line_join, line_width, name, subscribed_events, syncable, tags, x or y
Could not fit curve for exp_01_01 - CO2: unexpected attribute 'data_source' to Line, possible attributes are decorations, js_event_callbacks, js_property_callbacks, line_alpha, line_cap, line_color, line_dash, line_dash_offset, line_join, line_width, name, subscribed_events, syncable, tags, x or y
Could not fit curve for exp_01_02 - CH4: unexpected attribute 'data_source' to Line, possible attributes are decorations, js_event_callbacks, js_property_callbacks, line_alpha, line_cap, line_color, line_dash, line_dash_offset, line_join, line_width, name, subscribed_events, syncable

## 6) Actual plotting

In [8]:
# Pad with empty plots if we have fewer than 6 conditions
while len(plots) < 6:
    empty_plot = figure(width=400, height=300, title="No data")
    empty_plot.text([0.5], [0.5], text=["No data available"], 
                   text_align="center", text_baseline="middle")
    plots.append(empty_plot)

# Create 3x2 grid
grid = gridplot([plots[0:2], plots[2:4], plots[4:6]], 
                sizing_mode="scale_width")

show(grid)

## 7) Summarizing the performance of the model (time, r^2, projection at the end of the time period)

In [9]:
# Summary statistics and model performance
print("\n=== SUMMARY STATISTICS ===")
print(f"Total number of experiment conditions: {len(unique_conditions)}")
print(f"Time range: {min(time_points)} to {max(time_points)} days")
print(f"Gases measured: {', '.join(gases)}")

print("\n=== MODEL FITTING RESULTS ===")
for condition in selected_conditions:
    print(f"\nCondition: {condition}")
    condition_data = mean_data[mean_data['experiment_condition'] == condition]
    
    for gas in gases:
        gas_data = condition_data[condition_data['Gas'] == gas]
        
        if len(gas_data) >= 3:
            x_data = gas_data['days_since_start'].values
            y_data = gas_data['mean_conc'].values
            
            try:
                best_model, best_params, r2, model_name = fit_best_curve(x_data, y_data)
                
                if best_model is not None:
                    # Calculate projected concentration at day 60
                    future_conc = best_model(60, *best_params)
                    future_conc = max(future_conc, 0)  # No negative concentrations
                    
                    print(f"  {gas}: {model_name} model, R² = {r2:.3f}, Day 60 projection = {future_conc:.1f} ppm")
                else:
                    print(f"  {gas}: Could not fit curve")
            except Exception as e:
                print(f"  {gas}: Error in curve fitting - {e}")
        else:
            print(f"  {gas}: Insufficient data points ({len(gas_data)})")


=== SUMMARY STATISTICS ===
Total number of experiment conditions: 7
Time range: 0.0 to 42.0 days
Gases measured: CH4, CO2, N2O

=== MODEL FITTING RESULTS ===

Condition: exp_01_01
  CH4: linear model, R² = 0.982, Day 60 projection = 6021.6 ppm
  CO2: linear model, R² = 0.692, Day 60 projection = 22628.6 ppm
  N2O: Could not fit curve

Condition: exp_01_02
  CH4: logistic model, R² = 0.048, Day 60 projection = 45.3 ppm
  CO2: exponential model, R² = 0.836, Day 60 projection = 17005.6 ppm
  N2O: Could not fit curve

Condition: exp_01_03
  CH4: logistic model, R² = 0.957, Day 60 projection = 167.0 ppm
  CO2: logistic model, R² = 0.720, Day 60 projection = 16869.6 ppm
  N2O: Could not fit curve

Condition: exp_01_04
  CH4: logistic model, R² = 0.964, Day 60 projection = 133.8 ppm
  CO2: exponential model, R² = 0.788, Day 60 projection = 9069.9 ppm
  N2O: Could not fit curve

Condition: exp_01_05
  CH4: exponential model, R² = 0.996, Day 60 projection = 361.6 ppm
  CO2: linear model, R² = 

## 7) saving as a csv

In [10]:
# Export projection data to CSV for further analysis
projection_results = []

for condition in selected_conditions:
    condition_data = mean_data[mean_data['experiment_condition'] == condition]
    
    for gas in gases:
        gas_data = condition_data[condition_data['Gas'] == gas]
        
        if len(gas_data) >= 3:
            x_data = gas_data['days_since_start'].values
            y_data = gas_data['mean_conc'].values
            
            try:
                best_model, best_params, r2, model_name = fit_best_curve(x_data, y_data)
                
                if best_model is not None:
                    # Generate projections for standard time points
                    future_days = [42, 49, 56, 63, 70]  # Weekly projections
                    
                    for day in future_days:
                        projected_conc = max(best_model(day, *best_params), 0)
                        
                        projection_results.append({
                            'experiment_condition': condition,
                            'Gas': gas,
                            'projected_day': day,
                            'projected_concentration_ppm': projected_conc,
                            'model_type': model_name,
                            'r_squared': r2
                        })
            except:
                continue

# Create DataFrame and save
if projection_results:
    projection_df = pd.DataFrame(projection_results)
    projection_df.to_csv('gas_concentration_projections.csv', index=False)
    print(f"\nProjection results saved to 'gas_concentration_projections.csv'")
    print(f"Projection data shape: {projection_df.shape}")
    print("\nSample projections:")
    print(projection_df.head(10))
else:
    print("\nNo projection results to save")


Projection results saved to 'gas_concentration_projections.csv'
Projection data shape: (60, 6)

Sample projections:
  experiment_condition  Gas  projected_day  projected_concentration_ppm  \
0            exp_01_01  CH4             42                  4305.600559   
1            exp_01_01  CH4             49                  4972.918286   
2            exp_01_01  CH4             56                  5640.236013   
3            exp_01_01  CH4             63                  6307.553739   
4            exp_01_01  CH4             70                  6974.871466   
5            exp_01_01  CO2             42                 17724.258729   
6            exp_01_01  CO2             49                 19631.490674   
7            exp_01_01  CO2             56                 21538.722619   
8            exp_01_01  CO2             63                 23445.954564   
9            exp_01_01  CO2             70                 25353.186509   

  model_type  r_squared  
0     linear   0.981643  
1    

## 8) Unit Conversion and Alternative Visualization

Converting ppm concentrations to mol/L using astropy for unit conversions and creating additional plots with molar concentrations.

In [13]:
# 8a) Unit conversion functions using astropy
from astropy import units as u
from astropy import constants as const

# Define molar masses
molar_masses = {
    'CH4': 16.05 * u.g / u.mol,    # Methane
    'CO2': 44.01 * u.g / u.mol,    # Carbon dioxide
    'N2O': 46.01 * u.g / u.mol     # Nitrous oxide (note: you mentioned NO2 but data shows N2O)
}

def ppm_to_mol_per_L(concentration_ppm, gas_type, temperature=298.15*u.K, pressure=101325*u.Pa):
    """
    Convert ppm (parts per million by volume) to mol/L using astropy units
    
    For ideal gas: PV = nRT
    Concentration (mol/L) = P * (ppm/1e6) / (R * T)
    
    Parameters:
    - concentration_ppm: concentration in ppm
    - gas_type: 'CH4', 'CO2', or 'N2O'
    - temperature: temperature (default 298.15 K = 25°C)
    - pressure: pressure (default 101325 Pa = 1 atm)
    """
    # Convert ppm to fraction
    mole_fraction = concentration_ppm / 1e6
    
    # Use ideal gas law: n/V = P*χ/(R*T) where χ is mole fraction
    R = const.R  # Gas constant
    
    # Calculate molarity (mol/L)
    molarity = (pressure * mole_fraction / (R * temperature)).to(u.mol / u.L)
    
    return molarity.value

# Test the conversion
print("=== UNIT CONVERSION TEST ===")
for gas in gases:
    if gas in molar_masses:
        test_conc = 1000  # 1000 ppm
        mol_per_L = ppm_to_mol_per_L(test_conc, gas)
        print(f"{gas}: {test_conc} ppm = {mol_per_L:.6f} mol/L")

=== UNIT CONVERSION TEST ===
CH4: 1000 ppm = 0.000041 mol/L
CO2: 1000 ppm = 0.000041 mol/L
N2O: 1000 ppm = 0.000041 mol/L


In [14]:
# 8b) Convert existing data to mol/L
mean_data_molL = mean_data.copy()

# Add new columns for mol/L concentrations
mean_data_molL['mean_conc_molL'] = mean_data_molL.apply(
    lambda row: ppm_to_mol_per_L(row['mean_conc'], row['Gas']) if row['Gas'] in molar_masses else np.nan, 
    axis=1
)

mean_data_molL['std_conc_molL'] = mean_data_molL.apply(
    lambda row: ppm_to_mol_per_L(row['std_conc'], row['Gas']) if row['Gas'] in molar_masses and not pd.isna(row['std_conc']) else np.nan, 
    axis=1
)

print("=== CONVERTED DATA SAMPLE ===")
print("Original data (ppm) vs Converted data (mol/L):")
sample_data = mean_data_molL[['experiment_condition', 'Gas', 'mean_conc', 'mean_conc_molL']].head(10)
display(sample_data)

=== CONVERTED DATA SAMPLE ===
Original data (ppm) vs Converted data (mol/L):


,experiment_condition,Gas,mean_conc,mean_conc_molL
0,exp_01_01,CH4,51.330700,2.098093e-06
1,exp_01_01,CO2,3786.347480,1.547633e-04
2,exp_01_01,N2O,0.978991,4.001533e-08
3,exp_01_01,CH4,1129.136170,4.615236e-05
4,exp_01_01,CO2,8832.409226,3.610163e-04
5,exp_01_01,N2O,0.978997,4.001558e-08
6,exp_01_01,CH4,1719.985739,7.030277e-05
7,exp_01_01,CO2,8349.452822,3.412759e-04
8,exp_01_01,N2O,0.979877,4.005154e-08
9,exp_01_01,CH4,2388.532055,9.762897e-05


In [15]:
# 8c) Create plots with mol/L concentrations
def fit_best_curve_molL(x_data, y_data):
    """Same curve fitting function but for mol/L data"""
    return fit_best_curve(x_data, y_data)  # The math is the same, just different units

# Create new plots with mol/L units
plots_molL = []

for condition in selected_conditions:
    # Create figure for this condition with mol/L units
    p_molL = figure(
        title=f"Gas Concentrations - {condition} (mol/L)",
        x_axis_label="Days since start",
        y_axis_label="Concentration (mol/L)",
        width=400,
        height=300,
        tools="pan,wheel_zoom,box_zoom,reset,save"
    )
    
    # Add hover tool
    hover_molL = HoverTool(tooltips=[
        ("Gas", "@gas"),
        ("Day", "@x"),
        ("Concentration", "@y{0.000000} mol/L"),
        ("Model", "@model")
    ])
    p_molL.add_tools(hover_molL)
    
    # Filter data for this condition
    condition_data_molL = mean_data_molL[mean_data_molL['experiment_condition'] == condition]
    
    # Plot each gas
    for gas in gases:
        if gas in molar_masses:  # Only plot gases we can convert
            gas_data_molL = condition_data_molL[condition_data_molL['Gas'] == gas]
            
            if len(gas_data_molL) > 0:
                x_data = gas_data_molL['days_since_start'].values
                y_data = gas_data_molL['mean_conc_molL'].values
                y_err = gas_data_molL['std_conc_molL'].values
                
                # Plot observed data points
                observed_molL = p_molL.circle(
                    x_data, y_data,
                    size=8,
                    color=gas_colors[gas],
                    alpha=0.8,
                    legend_label=f"{gas} (observed)"
                )
                
                # Add error bars if available
                if not np.isnan(y_err).all():
                    p_molL.segment(x_data, y_data - y_err, x_data, y_data + y_err, 
                                 line_color=gas_colors[gas], line_alpha=0.5)
                
                # Fit curve and project
                if len(x_data) >= 3:  # Need at least 3 points for curve fitting
                    try:
                        best_model_molL, best_params_molL, r2_molL, model_name_molL = fit_best_curve_molL(x_data, y_data)
                        
                        if best_model_molL is not None:
                            # Generate projection
                            y_projection_molL = best_model_molL(projection_days, *best_params_molL)
                            
                            # Ensure no negative concentrations
                            y_projection_molL = np.maximum(y_projection_molL, 0)
                            
                            # Plot projection
                            projected_molL = p_molL.line(
                                projection_days, y_projection_molL,
                                line_color=gas_colors[gas],
                                line_dash='dashed',
                                line_width=2,
                                alpha=0.7,
                                legend_label=f"{gas} ({model_name_molL}, R²={r2_molL:.3f})"
                            )
                            
                    except Exception as e:
                        print(f"Could not fit curve for {condition} - {gas} (mol/L): {e}")
    
    # Customize legend
    p_molL.legend.location = "top_left"
    p_molL.legend.click_policy = "hide"
    p_molL.legend.label_text_font_size = "8pt"
    
    plots_molL.append(p_molL)

print(f"Created {len(plots_molL)} plots with mol/L units")

Created 6 plots with mol/L units


In [16]:
# 8d) Display the mol/L plots
# Pad with empty plots if we have fewer than 6 conditions
while len(plots_molL) < 6:
    empty_plot_molL = figure(width=400, height=300, title="No data")
    empty_plot_molL.text([0.5], [0.5], text=["No data available"], 
                        text_align="center", text_baseline="middle")
    plots_molL.append(empty_plot_molL)

# Create 3x2 grid for mol/L plots
grid_molL = gridplot([plots_molL[0:2], plots_molL[2:4], plots_molL[4:6]], 
                     sizing_mode="scale_width")

show(grid_molL)

## 9) Export Plots as HTML Files

Save interactive plots as standalone HTML files to access later

In [ ]:
# 9a) Import required Bokeh modules for HTML export
from bokeh.io import output_file, save
from datetime import datetime

# Create timestamp for unique filenames
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# 9b) Save PPM plots as HTML
output_file(f"gas_concentrations_ppm_{timestamp}.html")
save(grid)
print(f"PPM plots saved as: gas_concentrations_ppm_{timestamp}.html")

# 9c) Save mol/L plots as HTML
output_file(f"gas_concentrations_molL_{timestamp}.html")
save(grid_molL)
print(f"mol/L plots saved as: gas_concentrations_molL_{timestamp}.html")

print("\n=== HTML EXPORT COMPLETE ===")
print("Both sets of interactive plots have been saved as standalone HTML files.")
print("You can open these files in any web browser to view and interact with the plots.")
print("The files include:")
print(f"1. gas_concentrations_ppm_{timestamp}.html - Original plots in ppm units")
print(f"2. gas_concentrations_molL_{timestamp}.html - Converted plots in mol/L units")

In [ ]:
# 9d) Summary statistics for mol/L data
print("\n=== MOL/L UNIT SUMMARY ===")
print("Model fitting results with molar concentrations:")

for condition in selected_conditions:
    print(f"\nCondition: {condition}")
    condition_data_molL = mean_data_molL[mean_data_molL['experiment_condition'] == condition]
    
    for gas in gases:
        if gas in molar_masses:
            gas_data_molL = condition_data_molL[condition_data_molL['Gas'] == gas]
            
            if len(gas_data_molL) >= 3:
                x_data = gas_data_molL['days_since_start'].values
                y_data = gas_data_molL['mean_conc_molL'].values
                
                try:
                    best_model_molL, best_params_molL, r2_molL, model_name_molL = fit_best_curve_molL(x_data, y_data)
                    
                    if best_model_molL is not None:
                        # Calculate projected concentration at day 60 in mol/L
                        future_conc_molL = best_model_molL(60, *best_params_molL)
                        future_conc_molL = max(future_conc_molL, 0)  # No negative concentrations
                        
                        print(f"  {gas}: {model_name_molL} model, R² = {r2_molL:.3f}, Day 60 projection = {future_conc_molL:.6f} mol/L")
                    else:
                        print(f"  {gas}: Could not fit curve")
                except Exception as e:
                    print(f"  {gas}: Error in curve fitting - {e}")
            else:
                print(f"  {gas}: Insufficient data points ({len(gas_data_molL)})")

print(f"\n=== CONVERSION FACTORS USED ===")
for gas, molar_mass in molar_masses.items():
    print(f"{gas}: Molar mass = {molar_mass}")

print(f"\nConversion performed using ideal gas law at STP conditions:")
print(f"Temperature = 298.15 K (25°C)")
print(f"Pressure = 1 atm") 
print(f"mol/L = (P × ppm/1e6) / (R × T)")